# 1. 패키지 설치

In [1]:
%pip install langchain langchain-core langchain-community langchain-text-splitters langchain-openai langchain-pinecone docx2txt

   ---------------------------------------- 0.0/587.6 kB ? eta -:--:--
   ---------------------------------------- 587.6/587.6 kB 7.9 MB/s eta 0:00:00

   ------ --------------------------------- 1/6 [pinecone-plugin-interface]
   ------------- -------------------------- 2/6 [pinecone-plugin-assistant]
   ------------- -------------------------- 2/6 [pinecone-plugin-assistant]
   ------------- -------------------------- 2/6 [pinecone-plugin-assistant]
   ------------- -------------------------- 2/6 [pinecone-plugin-assistant]
   ------------- -------------------------- 2/6 [pinecone-plugin-assistant]
   -------------------- ------------------- 3/6 [pinecone]
   -------------------- ------------------- 3/6 [pinecone]
   -------------------- ------------------- 3/6 [pinecone]
   -------------------- ------------------- 3/6 [pinecone]
   -------------------- ------------------- 3/6 [pinecone]
   -------------------- ------------------- 3/6 [pinecone]
   -------------------- --------------


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
%pip install pinecone

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Knowledge Base 구성을 위한 데이터 생성

- Chroma를 활용한 [3.2 LangChain과 Chroma를 활용한 RAG 구성](https://github.com/jasonkang14/inflearn-rag-notebook/blob/main/3.2%20LangChain%EA%B3%BC%20Chroma%EB%A5%BC%20%ED%99%9C%EC%9A%A9%ED%95%9C%20RAG%20%EA%B5%AC%EC%84%B1.ipynb)과 동일함
- Vector Database만 [Pinecone](https://www.pinecone.io/)으로 변경

In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax_with_markdown.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [5]:
document_list[52]

Document(metadata={'source': './tax_with_markdown.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과        | 3억 8,406만원 + (10억원을 초과하는 금액의 45퍼센트)|\n\n\n\n\n\n② 거주자의 퇴직소득에 대한 소득세는 다음 각 호의 순서에 따라 계산한 금액(이하 “퇴직소득 산출세액”이라 한

In [18]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [19]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'tax-index'
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

In [12]:
query = '연봉 5천만원인 거주자의 종합소득세는?'

# 3. 답변 생성을 위한 Retrieval

- `RetrievalQA`에 전달하기 위해 `retriever` 생성
- `search_kwargs` 의 `k` 값을 변경해서 가져올 문서의 갯수를 지정할 수 있음
- `.invoke()` 를 호출해서 어떤 문서를 가져오는지 확인 가능

In [13]:
retriever = database.as_retriever(search_kwargs={'k': 4})
retriever.invoke(query)

[Document(id='5ca6a51b-9c8a-4d7c-8910-4d3a82d7cacb', metadata={'source': './tax_with_markdown.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과        | 3억 8,406만원 + (10억원을 초과하는 금액의 45퍼센트)|\n\n\n\n\n\n② 거주자의 퇴직소득에 대한 소

[Document(id='5ca6a51b-9c8a-4d7c-8910-4d3a82d7cacb', metadata={'source': './tax_with_markdown.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 7,406만원 + (5억원을 초과하는 금액의 42퍼센트)|\n\n| 10억원 초과        | 3억 8,406만원 + (10억원을 초과하는 금액의 45퍼센트)|\n\n\n\n\n\n② 거주자의 퇴직소득에 대한 소득세는 다음 각 호의 순서에 따라 계산한 금액(이하 “퇴직소득 산출세액”이라 한다)으로 한다.<개정 2013. 1. 1., 2014. 12. 23.>\n\n1. 해당 과세기간의 퇴직소득과세표준에 제1항의 세율을 적용하여 계산한 금액\n\n2. 제1호의 금액을 12로 나눈 금액에 근속연수를 곱한 금액\n\n3. 삭제<2014. 12. 23.>\n\n[전문개정 2009. 12. 31.]\n\n\n\n제2관 세액공제 <개정 2009. 12. 31.>\n\n\n\n제56조(배당세액공제) ① 거주자의 종합소득금액에 제17조제3항 각 호 외의 부분 단서가 적용되는 배당소득금액이 합산되어 있는 경우에는 같은 항 각 호 외의 부분 단서에 따라 해당 과세기간의 총수입금액에 더한 금액에 해당하는 금액을 종합소득 산출세액에서 공제한다. <개정 2009. 12. 31.>\n\n② 제1항에 따른 공제를 “배당세액공제”라 한다.<개정 2009. 12. 31.>\n\n③ 삭제<2003. 12. 30.>\n\n④ 제1항을 적용할 때 배당세액공제의 대상이 되는 배당소득금액은 제14조제2항의 종합소득과세표준에 포함된 배당소득금액으로서 이자소득등의 종합과세기준금액을 초과하는 것으로 한다.<개정 2009. 12. 31.>\n\n⑤ 삭제<2006. 12. 30.>'),
 Document(id='d4c17380-ad68-465f-a26d-fbe71d572251', metadata={'source': './tax_with_markdown.docx'}, page_content='3. 「자본시장과 금융투자업에 관한 법률」 제251조제1항에 따른 집합투자업겸영보험회사의 특별계정\n\n③ 비거주자의 소득은 제119조에 따라 구분한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제4조(소득의 구분) ① 거주자의 소득은 다음 각 호와 같이 구분한다. <개정 2013. 1. 1., 2020. 12. 29.>\n\n1. 종합소득\n\n\u3000이 법에 따라 과세되는 모든 소득에서 제2호, 제2호의2 및 제3호에 따른 소득을 제외한 소득으로서 다음 각 목의 소득을 합산한 것\n\n가. 이자소득\n\n나. 배당소득\n\n다. 사업소득\n\n라. 근로소득\n\n마. 연금소득\n\n바. 기타소득\n\n2. 퇴직소득\n\n2의2. 금융투자소득\n\n3. 양도소득\n\n② 제1항에 따른 소득을 구분할 때 다음 각 호의 신탁을 제외한 신탁의 이익은 「신탁법」 제2조에 따라 수탁자에게 이전되거나 그 밖에 처분된 재산권에서 발생하는 소득의 내용별로 구분한다.<개정 2011. 7. 25., 2020. 12. 29., 2022. 12. 31.>\n\n1. 「법인세법」 제5조제2항에 따라 신탁재산에 귀속되는 소득에 대하여 그 신탁의 수탁자가 법인세를 납부하는 신탁\n\n2. 「자본시장과 금융투자업에 관한 법률」 제9조제18항제1호에 따른 투자신탁. 다만, 2024년 12월 31일까지는 이 법 제17조제1항제5호에 따른 집합투자기구로 한정한다.\n\n3. 「자본시장과 금융투자업에 관한 법률」 제251조제1항에 따른 집합투자업겸영보험회사의 특별계정\n\n③ 비거주자의 소득은 제119조에 따라 구분한다.\n\n[전문개정 2009. 12. 31.]\n\n[시행일: 2025. 1. 1.] 제4조제1항제1호, 제4조제1항제2호의2\n\n\n\n제5조(과세기간) ① 소득세의 과세기간은 1월 1일부터 12월 31일까지 1년으로 한다.\n\n② 거주자가 사망한 경우의 과세기간은 1월 1일부터 사망한 날까지로 한다.\n\n③ 거주자가 주소 또는 거소를 국외로 이전(이하 “출국”이라 한다)하여 비거주자가 되는 경우의 과세기간은 1월 1일부터 출국한 날까지로 한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제6조(납세지) ① 거주자의 소득세 납세지는 그 주소지로 한다. 다만, 주소지가 없는 경우에는 그 거소지로 한다.\n\n② 비거주자의 소득세 납세지는 제120조에 따른 국내사업장(이하 “국내사업장”이라 한다)의 소재지로 한다. 다만, 국내사업장이 둘 이상 있는 경우에는 주된 국내사업장의 소재지로 하고, 국내사업장이 없는 경우에는 국내원천소득이 발생하는 장소로 한다.<개정 2013. 1. 1.>\n\n③ 납세지가 불분명한 경우에는 대통령령으로 정하는 바에 따라 납세지를 결정한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제7조(원천징수 등의 경우의 납세지) ① 원천징수하는 소득세의 납세지는 다음 각 호에 따른다. <개정 2012. 1. 1., 2023. 12. 31.>'),
 Document(id='0138c46d-a03e-48d3-b548-c4d013bf8632', metadata={'source': './tax_with_markdown.docx'}, page_content='③ 거주자의 부양가족 중 거주자(그 배우자를 포함한다)의 직계존속이 주거 형편에 따라 별거하고 있는 경우에는 제1항에도 불구하고 제50조에서 규정하는 생계를 같이 하는 사람으로 본다.\n\n④ 제50조, 제51조 및 제59조의2에 따른 공제대상 배우자, 공제대상 부양가족, 공제대상 장애인 또는 공제대상 경로우대자에 해당하는지 여부의 판정은 해당 과세기간의 과세기간 종료일 현재의 상황에 따른다. 다만, 과세기간 종료일 전에 사망한 사람 또는 장애가 치유된 사람에 대해서는 사망일 전날 또는 치유일 전날의 상황에 따른다.<개정 2014. 1. 1.>\n\n⑤ 제50조제1항제3호 및 제59조의2에 따라 적용대상 나이가 정해진 경우에는 제4항 본문에도 불구하고 해당 과세기간의 과세기간 중에 해당 나이에 해당되는 날이 있는 경우에 공제대상자로 본다.<개정 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n\n\n제54조(종합소득공제 등의 배제) ① 분리과세이자소득, 분리과세배당소득, 분리과세연금소득과 분리과세기타소득만이 있는 자에 대해서는 종합소득공제를 적용하지 아니한다. <개정 2013. 1. 1.>\n\n② 제70조제1항, 제70조의2제2항 또는 제74조에 따라 과세표준확정신고를 하여야 할 자가 제70조제4항제1호에 따른 서류를 제출하지 아니한 경우에는 기본공제 중 거주자 본인에 대한 분(分)과 제59조의4제9항에 따른 표준세액공제만을 공제한다. 다만, 과세표준확정신고 여부와 관계없이 그 서류를 나중에 제출한 경우에는 그러하지 아니하다.<개정 2013. 1. 1., 2014. 1. 1.>\n\n③ 제82조에 따른 수시부과 결정의 경우에는 기본공제 중 거주자 본인에 대한 분(分)만을 공제한다.\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>'),
 Document(id='9e10523f-4825-487f-8b34-cbe8e818831d', metadata={'source': './tax_with_markdown.docx'}, page_content='⑤ 공동으로 소유한 자산에 대한 양도소득금액을 계산하는 경우에는 해당 자산을 공동으로 소유하는 각 거주자가 납세의무를 진다.<신설 2017. 12. 19., 2020. 12. 29.>\n\n[전문개정 2009. 12. 31.]\n\n[제2조에서 이동 <2009. 12. 31.>]\n\n\n\n제2조의3(신탁재산 귀속 소득에 대한 납세의무의 범위) ① 신탁재산에 귀속되는 소득은 그 신탁의 이익을 받을 수익자(수익자가 사망하는 경우에는 그 상속인)에게 귀속되는 것으로 본다.\n\n② 제1항에도 불구하고 위탁자가 신탁재산을 실질적으로 통제하는 등 대통령령으로 정하는 요건을 충족하는 신탁의 경우에는 그 신탁재산에 귀속되는 소득은 위탁자에게 귀속되는 것으로 본다.<개정 2023. 12. 31.>\n\n[본조신설 2020. 12. 29.]\n\n\n\n제3조(과세소득의 범위) ① 거주자에게는 이 법에서 규정하는 모든 소득에 대해서 과세한다. 다만, 해당 과세기간 종료일 10년 전부터 국내에 주소나 거소를 둔 기간의 합계가 5년 이하인 외국인 거주자에게는 과세대상 소득 중 국외에서 발생한 소득의 경우 국내에서 지급되거나 국내로 송금된 소득에 대해서만 과세한다.\n\n② 비거주자에게는 제119조에 따른 국내원천소득에 대해서만과세한다.\n\n③ 제1항 및 제2항을 적용하는 경우 「조세특례제한법」 제100조의14제2호의 동업자에게는 같은 법 제100조의18제1항에 따라 배분받은 소득 및 같은 법 제100조의22제1항에 따라 분배받은 자산의 시가 중 분배일의 지분가액을 초과하여 발생하는 소득에 대하여 과세한다.\n\n[전문개정 2009. 12. 31.]\n\n\n\n제4조(소득의 구분) ① 거주자의 소득은 다음 각 호와 같이 구분한다. <개정 2013. 1. 1.>\n\n1. 종합소득\n\n\u3000이 법에 따라 과세되는 모든 소득에서 제2호 및 제3호에 따른 소득을 제외한 소득으로서 다음 각 목의 소득을 합산한 것\n\n가. 이자소득\n\n나. 배당소득\n\n다. 사업소득\n\n라. 근로소득\n\n마. 연금소득\n\n바. 기타소득\n\n2. 퇴직소득\n\n3. 양도소득\n\n② 제1항에 따른 소득을 구분할 때 다음 각 호의 신탁을 제외한 신탁의 이익은 「신탁법」 제2조에 따라 수탁자에게 이전되거나 그 밖에 처분된 재산권에서 발생하는 소득의 내용별로 구분한다.<개정 2011. 7. 25., 2020. 12. 29., 2022. 12. 31.>\n\n1. 「법인세법」 제5조제2항에 따라 신탁재산에 귀속되는 소득에 대하여 그 신탁의 수탁자가 법인세를 납부하는 신탁\n\n2. 「자본시장과 금융투자업에 관한 법률」 제9조제18항제1호에 따른 투자신탁. 다만, 2024년 12월 31일까지는 이 법 제17조제1항제5호에 따른 집합투자기구로 한정한다.\n\n3. 「자본시장과 금융투자업에 관한 법률」 제251조제1항에 따른 집합투자업겸영보험회사의 특별계정\n\n③ 비거주자의 소득은 제119조에 따라 구분한다.\n\n[전문개정 2009. 12. 31.]')]

# 4. Augmentation을 위한 Prompt 활용

- Retrieval된 데이터는 LangChain에서 제공하는 프롬프트(`"rlm/rag-prompt"`) 사용

In [20]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt("rlm/rag-prompt")

In [21]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

# 5. 답변 생성

- [RetrievalQA](https://docs.smith.langchain.com/old/cookbook/hub-examples/retrieval-qa-chain)를 통해 LLM에 전달
    - `RetrievalQA`는 [create_retrieval_chain](https://python.langchain.com/v0.2/docs/how_to/qa_sources/#using-create_retrieval_chain)으로 대체됨
    - 실제 ChatBot 구현 시 `create_retrieval_chain`으로 변경하는 과정을 볼 수 있음

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# 문서 리스트 → 문자열 변환 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt
prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context.\n\n"
    "Context:\n{context}\n\n"
    "Question:\n{question}\n\n"
    "Answer in Korean."
)

# Retriever
#retriever = database.as_retriever(search_kwargs={"k": 3})

# 체인
qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

response = qa_chain.invoke(query)

print(response)

content='연봉 5천만원인 거주자의 종합소득세는 다음과 같이 계산할 수 있습니다.\n\n- 1,400만원 이하의 부분에 대해서는 6%의 세율이 적용됩니다.\n- 1,400만원을 초과하고 5,000만원 이하의 부분에 대해서는 84만원 + (1,400만원을 초과하는 금액의 15%)가 적용됩니다.\n\n계산 과정은 다음과 같습니다:\n\n1. 1,400만원 부분에 대한 세금: 1,400만원 × 6% = 84만원\n2. 1,400만원을 초과하는 3,600만원 부분에 대한 세금: 3,600만원 × 15% = 540만원\n\n따라서 총 종합소득세는 84만원 + 540만원 = 624만원입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 3771, 'total_tokens': 3955, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_fab7bd3a94', 'id': 'chatcmpl-DbgGCKkkTflyalZlfL2wde5qhUXNo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019df17c-6231-74a2-a267-f8a6b56df397-0' tool_calls=[] invalid_tool_calls=[] usage_metad

content='연봉 5천만원인 거주자의 종합소득세는 다음과 같이 계산할 수 있습니다.\n\n- 1,400만원 이하의 부분에 대해서는 6%의 세율이 적용됩니다.\n- 1,400만원을 초과하고 5,000만원 이하의 부분에 대해서는 84만원 + (1,400만원을 초과하는 금액의 15%)가 적용됩니다.\n\n계산 과정은 다음과 같습니다:\n\n1. 1,400만원 부분에 대한 세금: 1,400만원 × 6% = 84만원\n2. 1,400만원을 초과하는 3,600만원 부분에 대한 세금: 3,600만원 × 15% = 540만원\n\n따라서 총 종합소득세는 84만원 + 540만원 = 624만원입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 3771, 'total_tokens': 3955, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_fab7bd3a94', 'id': 'chatcmpl-DbgGCKkkTflyalZlfL2wde5qhUXNo', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019df17c-6231-74a2-a267-f8a6b56df397-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 3771, 'output_tokens': 184, 'total_tokens': 3955, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}